In [ ]:
!pip install -q pandas requests groq sentence-transformers faiss-cpu Sastrawi openpyxl fastapi uvicorn pyngrok nest_asyncio python-dotenv
!pip install groq

import os
import re
import json
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from groq import Groq
from sentence_transformers import SentenceTransformer
import faiss
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("✅ Packages installed!")

In [ ]:
import os
import sys
from dotenv import load_dotenv
from google.colab import files
import nest_asyncio

print("Langkah Keamanan: Mohon upload file .env anda terlebih dahulu.")

if os.path.exists(".env"):
    os.remove(".env")

uploaded = files.upload()

if uploaded:
    uploaded_filename = next(iter(uploaded))

    content = uploaded[uploaded_filename]
    with open(".env", "wb") as f:
        f.write(content)

    # Load variable
    load_dotenv()

    # Verifikasi Key
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

    if GROQ_API_KEY:
        # Set environment variable explisit untuk Groq
        os.environ["GROQ_API_KEY"] = GROQ_API_KEY
        print(f"✅ SUKSES! File '{uploaded_filename}' berhasil diproses jadi .env")
        print(f"🔑 Key terdeteksi: {GROQ_API_KEY[:5]}... (Aman)")
    else:
        print("⚠️ WARNING: File berhasil diupload, tapi GROQ_API_KEY tidak terbaca. Cek isi file .env anda!")
else:
    sys.exit("❌ ERROR: Tidak ada file yang diupload. Program dihentikan.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path config
BASE_DIR = "/content/drive/MyDrive/capstonelb/"
EXCEL_LP = BASE_DIR + "LP and Course Mapping.xlsx"
EXCEL_RB = BASE_DIR + "Resource Data Learning Buddy.xlsx"
SAVE_PATH = BASE_DIR + "learning_buddy_model/"

# API Keys
DICODING_API_BASE = "https://jrkqcbmjknzgpbtrupxh.supabase.co/rest/v1"
DICODING_API_KEY = "sb_publishable_h889CjrPIGwCMA9I4oTTaA_2L22Y__R"
DICODING_HEADERS = {
    "apikey": DICODING_API_KEY,
    "Authorization": f"Bearer {DICODING_API_KEY}"
}

EMBEDDING_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

print("✅ Config loaded!")

In [ ]:
print("📦 Loading Excel data...")

xls_lp = pd.ExcelFile(EXCEL_LP)
xls_rb = pd.ExcelFile(EXCEL_RB)

df_learning_paths = pd.read_excel(xls_lp, sheet_name="Learning Path")
df_courses = pd.read_excel(xls_lp, sheet_name="Course")
df_tutorials = pd.read_excel(xls_lp, sheet_name="Tutorials")
df_course_levels = pd.read_excel(xls_lp, sheet_name="Course Level")
df_lp_mapping = pd.read_excel(xls_lp, sheet_name="LP + Course")

df_interest_questions = pd.read_excel(xls_rb, sheet_name="Current Interest Questions")
df_tech_questions = pd.read_excel(xls_rb, sheet_name="Current Tech Questions")
df_skill_keywords = pd.read_excel(xls_rb, sheet_name="Skill Keywords")
# PERBAIKI: Pastikan nama kolom konsisten
if "keyword" in df_skill_keywords.columns:
    df_skill_keywords = df_skill_keywords.rename(columns={"keyword": "skill"})

print(f"✅ Skill keywords loaded: {len(df_skill_keywords)} items")
print(f"   Columns: {df_skill_keywords.columns.tolist()}")
df_student_progress = pd.read_excel(xls_rb, sheet_name="Student Progress")

try:
    df_quiz_questions = pd.read_excel(xls_rb, sheet_name="Current Tech Questions")
    print("✅ Quiz Questions loaded!")
except:
    print("⚠️ Warning: Sheet 'Current Tech Questions' not found. Please check Excel file.")
    # Fallback kosong biar ga crash
    df_quiz_questions = pd.DataFrame(columns=['tech_category', 'difficulty', 'question_desc', 'option_1', 'option_2', 'option_3', 'option_4', 'correct_answer'])

print("✅ Excel data loaded!")

In [ ]:
print("🔍 Loading saved model...")

try:
    df_full = pd.read_pickle(SAVE_PATH + "df_full.pkl")
    faiss_index = faiss.read_index(SAVE_PATH + "faiss_index.bin")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print(f"✅ df_full loaded: {len(df_full)} tutorials")
    print(f"✅ FAISS index loaded: {faiss_index.ntotal} vectors")
    print("✅ Sentence Transformer model loaded!")

except Exception as e:
    print(f"❌ ERROR: {e}")
    print(f"Pastikan file ada di: {SAVE_PATH}")

In [ ]:
# Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = stemmer.stem(text)
    return text

print("✅ Text cleaning ready!")

In [ ]:
def search_tutorials(query, top_k=5):
    """Search tutorials menggunakan semantic similarity"""
    query_clean = clean_text(query)
    query_emb = model.encode([query_clean]).astype('float32')

    distances, indices = faiss_index.search(query_emb, top_k * 2)

    results = []
    seen_titles = set()

    for idx in indices[0]:
        if idx >= len(df_full):
            continue
        row = df_full.iloc[idx]
        title = row['tutorial_title']

        if title in seen_titles or pd.isna(title):
            continue

        seen_titles.add(title)
        results.append({
            'tutorial': title,
            'course': row.get('course_name', ''),
            'learning_path': row.get('learning_path_name', ''),
            'level': row.get('course_level', ''),
            'course_id': int(row.get('course_id', 0))
        })

        if len(results) >= top_k:
            break

    return results

print("✅ Search function ready!")

In [ ]:
import re

LEVEL_KEYWORDS = {
    'Beginner': ['pemula', 'baru mulai', 'belum paham', 'dasar', 'belajar dari nol', 'awam', 'newbie'],
    'Intermediate': ['cukup paham', 'menengah', 'sudah paham', 'pernah belajar', 'lumayan', 'sedang'],
    'Advanced': ['mahir', 'expert', 'advanced', 'berpengalaman', 'profesional', 'jago', 'pro']
}

ROLE_PATTERNS = {
    'AI Engineer': ['ai', 'machine learning', 'deep learning', 'ml engineer', 'data science', 'artificial intelligence'],
    'Android Developer': ['android', 'mobile', 'kotlin', 'jetpack', 'aplikasi mobile'],
    'Backend Developer': ['backend', 'back-end', 'server', 'api', 'database', 'rest api'],
    'Frontend Developer': ['frontend', 'front-end', 'web', 'html', 'css', 'javascript', 'react', 'vue'],
    'DevOps Engineer': ['devops', 'cloud', 'kubernetes', 'docker', 'ci/cd', 'deployment'],
    'Data Scientist': ['data scientist', 'data analyst', 'data science', 'analytics', 'visualization'],
    'Full Stack Developer': ['full stack', 'fullstack', 'full-stack', 'frontend backend']
}

def detect_level(text):
    text_lower = text.lower()
    for level, keywords in LEVEL_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return level
    return 'Beginner'

def detect_role(text):
    text_lower = text.lower()
    scores = {}
    for role, patterns in ROLE_PATTERNS.items():
        score = sum(1 for pattern in patterns if pattern in text_lower)
        if score > 0:
            scores[role] = score
    if scores:
        return max(scores, key=scores.get)
    return None

def extract_skills(text):
    """
    LAYER 2: Extract technical skills - FIXED & ANTI-HALU VERSION
    """
    text_lower = text.lower()
    detected = []

    BLACKLIST_WORDS = {
        # Kata kerja/sifat umum
        'belajar', 'mulai', 'ingin', 'mau', 'bisa', 'dapat', 'punya', 'pakai',
        'suka', 'coba', 'lagi', 'sedang', 'akan', 'sudah', 'belum', 'masih',
        'skill', 'kemampuan', 'hal', 'cara', 'tips', 'saran', 'bantuan',
        'dasar', 'basic', 'bagus', 'baik', 'jelek', 'susah', 'mudah',
        'aku', 'saya', 'kamu', 'dia', 'kita', 'mereka',
        'dong', 'nih', 'ya', 'kan', 'sih', 'deh', 'kok', 'gimana',
        'untuk', 'pada', 'dengan', 'dari', 'ini', 'itu',
        # Level words
        'pemula', 'beginner', 'intermediate', 'advanced', 'mahir',
        'menengah', 'expert', 'pro', 'newbie',
        # 🚨 KATA JEBAKAN (TESTING)
        'test', 'tes', 'cek', 'testing', 'coba', 'halo', 'hai', 'ping', 'p'
    }

    keyword_col = "skill" if "skill" in df_skill_keywords.columns else "keyword"

    for kw in df_skill_keywords[keyword_col].astype(str).str.lower().tolist():
        kw = kw.strip()

        # Skip blacklisted words
        if kw in BLACKLIST_WORDS or len(kw) < 2:
            continue

        is_short_tech = len(kw) <= 3 and kw in ['c', 'r', 'go', 'js', 'ai', 'ml']
        is_normal_tech = len(kw) > 3

        if not (is_short_tech or is_normal_tech):
            continue

        # Match in text
        if ' ' in kw:
            if kw in text_lower:
                detected.append(kw)
        else:
            pattern = r'\b' + re.escape(kw) + r'\b'
            if re.search(pattern, text_lower):
                detected.append(kw)

    return list(set(detected))

def assess_user(user_input):
    return {
        "role": detect_role(user_input),
        "level": detect_level(user_input),
        "skills": extract_skills(user_input)
    }

print("✅ Multi-layer assessment (Anti-Halu) ready!")

In [ ]:
def build_context_for_llama(user_input, assessment, tutorials, roadmap, user_history=None):
    """
    Build context untuk Llama 3
    Sesuai soal: Learning Buddy sebagai personal assistant & teman diskusi
    """

    skills = assessment.get('skills', [])
    skills_text = ', '.join(skills[:5]) if skills else 'Belum terdeteksi'

    context = f"""Kamu adalah Learning Buddy, personal learning assistant dari Dicoding.

PERSONALITY & ROLE (sesuai soal):
- Teman belajar yang supportive dan bisa diajak diskusi kapan saja
- Bukan cuma recommender bot, tapi personal assistant yang paham progress user
- Berikan motivasi & saran strategi belajar
- Natural dan casual tapi tetap helpful
- HANYA pakai "Halo/Hi" sekali di awal conversation

MULTI-LAYER ASSESSMENT (sesuai soal):
- Layer 1 (Job Role): {assessment.get('role', 'Belum terdeteksi')}
- Layer 2 (Skills Extracted): {skills_text}
- Layer 3 (Level): {assessment.get('level', 'Beginner')}

PRIORITY RECOMMENDATION:
1. UTAMAKAN course/tutorial dari Dicoding (data di bawah)
2. JIKA data Dicoding kurang/tidak ada: BOLEH suggest external resources
3. Sebutkan dengan jelas mana dari Dicoding, mana external
"""

    # User history (untuk progress tracking)
    if user_history:
        context += f"""

PROGRESS TRACKING (sesuai soal: jawab pertanyaan progress):
- Nama: {user_history.get('name', '')}
- Course aktif: {user_history.get('current_course', '')}
- Tutorial selesai: {user_history.get('completed_tutorials', 0)}/{user_history.get('active_tutorials', 0)}
- Progress: {user_history.get('progress_percentage', 0)}%
- Sisa tutorial: {user_history.get('tutorials_remaining', 0)}
- Target berikutnya: {user_history.get('next_milestone', '')}
- Skill yang berkembang: Analisis dari progress di atas

INSIGHT UNTUK USER (sesuai soal):
- Berikan insight personal: "skill mana yang sudah berkembang?"
- Saran strategi: "bagaimana cara meningkatkan pembelajaran?"
- Motivasi: sesuai progress mereka
"""

    # Dicoding tutorials
    if tutorials and len(tutorials) > 0:
        context += f"""

TUTORIAL DICODING (PRIORITAS UTAMA):
"""
        for i, tut in enumerate(tutorials[:5], 1):
            context += f"{i}. {tut['tutorial']} - {tut['course']} ({tut['level']})\n"
    else:
        context += "\n⚠️ TIDAK ADA tutorial Dicoding yang match.\n"

    # Roadmap
    if roadmap and len(roadmap) > 0:
        context += f"""

ROADMAP BELAJAR (dari Dicoding):
"""
        for item in roadmap[:6]:
            context += f"Week {item['week']}: {item['course']} ({item['level']})\n"
    else:
        context += "\n⚠️ TIDAK ADA roadmap Dicoding yang match.\n"

    # External resources (BOLEH - sesuai soal)
    if should_use_external_resources(tutorials, roadmap):
        ext_resources = get_external_resources(assessment.get('role', '') or (skills[0] if skills else ''))
        if ext_resources:
            context += f"""

EXTERNAL RESOURCES (karena data Dicoding terbatas):
"""
            for i, res in enumerate(ext_resources, 1):
                context += f"{i}. {res['title']} ({res['type']}) - {res['url']}\n"

            context += "\n💡 Catatan: Sebutkan ke user bahwa ini resource pelengkap di luar Dicoding.\n"

    context += f"""

USER MESSAGE: "{user_input}"

INSTRUKSI JAWABAN (sesuai soal):
1. Jadilah teman diskusi yang bisa diajak bicara kapan saja
2. Berikan motivasi & saran strategi belajar personal
3. Jawab pertanyaan progress dengan insight detail
4. Prioritas: Dicoding → External (jika perlu)
5. Tidak ngelantur, tapi tetap natural
6. Maksimal 300 kata
7. Gunakan emoji secukupnya

Jawab sekarang sebagai Learning Buddy yang supportive:
"""

    return context

print("✅ Context builder ready (with external resources option)!")

In [ ]:
def format_progress_message(progress_data):
    """Format progress data menjadi message untuk Gemini"""
    if not progress_data:
        return "User belum memiliki data progress."

    return f"""
Progress {progress_data['name']}:
- Course: {progress_data['course']}
- Tutorial aktif: {progress_data['active_tutorials']}
- Tutorial selesai: {progress_data['completed_tutorials']}
- Sudah lulus: {'Ya' if progress_data['is_graduated'] else 'Belum'}
- Nilai exam: {progress_data['exam_score']}
- Rating submission: {progress_data['submission_rating']}
"""

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. HELPER PROGRESS (LENGKAP & RAPI)
# ==========================================
def get_detailed_history(email):
    """
    ✅ GRACEFUL HANDLING: Anti-Crash & Safe Data
    """
    if not email: return None
    if 'df_student_progress' not in globals(): return None

    try:
        mask = df_student_progress['email'].astype(str).str.lower().str.strip() == str(email).lower().strip()
        user_data = df_student_progress[mask]
    except: return None

    if user_data.empty: return None
    user = user_data.iloc[0]

    def safe_get(col_name, default_val, data_type=str):
        try:
            val = user.get(col_name)
            if pd.isna(val) or val is None or str(val).strip() == "": return default_val
            if data_type == int: return int(float(val))
            if data_type == float: return float(val)
            if data_type == bool: return bool(val)
            return str(val)
        except: return default_val

    data = {
        "name": safe_get("name", "User", str),
        "course": safe_get("course_name", "-", str),
        "active": safe_get("active_tutorials", 1, int),
        "completed": safe_get("completed_tutorials", 0, int),
        "graduated": safe_get("is_graduated", 0, int),
        "exam": safe_get("exam_score", 0, float),
        "rating": safe_get("submission_rating", 0, float),
        "certificate": safe_get("already_generated_certificate", 0, int),
    }

    if data["active"] <= 0: data["active"] = 1
    display_completed = min(data["completed"], data["active"])
    pct = round((display_completed / data["active"]) * 100, 1)

    if data["graduated"]: milestone = "🎉 Lulus! Next course?"
    elif pct < 50: milestone = "Yuk kejar 50%!"
    else: milestone = "Semangat lanjutin!"

    return {
        "name": data["name"],
        "course": data["course"],
        "completed": display_completed,
        "total": data["active"],
        "pct": pct,
        "is_graduated": data["graduated"],
        "exam": data["exam"],
        "rating": data["rating"],
        "certificate": bool(data["certificate"]),
        "milestone": milestone
    }

def format_detailed_history(h):
    if not h: return "Data progress tidak ditemukan."

    if h['is_graduated']:
        status_icon = "LULUS 🎓"
        cert_str = "✅ Sertifikat Ready" if h['certificate'] else "⏳ Sertifikat Proses"
    else:
        status_icon = "ON GOING 🚀"
        cert_str = ""

    score_str = ""
    if h['exam'] > 0: score_str += f"\n📝 Exam: {h['exam']}"
    if h['rating'] > 0: score_str += f" | ⭐ Rating: {h['rating']}/5"

    # PEMISAH PENDEK BIAR RAPI DI WEB
    separator = "=" * 20

    return f"""👤 **{h['name']}**
📊 **PROGRESS UPDATE**
{separator}
🎯 **{h['course']}**
{status_icon} | 🔥 {h['pct']}% Selesai

✅ Capaian: {h['completed']}/{h['total']} Modul
{cert_str}{score_str}
📌 Target: {h['milestone']}"""

In [ ]:
import random
import textwrap

# ==========================================
# 1. CLASS QUIZ (JANGAN DIUBAH)
# ==========================================
class QuizSession:
    def __init__(self, topic=None):
        self.questions = []
        self.current_index = 0
        self.score = 0
        self.answers_map = {}
        self.active = False
        self.topic = topic

    def start(self, df_questions, specific_topic=None):
        if specific_topic:
            filtered = df_questions[df_questions['tech_category'].str.contains(specific_topic, case=False, na=False)]
            pool = filtered if len(filtered) >= 10 else df_questions
        else:
            pool = df_questions

        sample_size = min(10, len(pool))
        self.questions = pool.sample(n=sample_size).to_dict('records')
        self.current_index = 0
        self.score = 0
        self.active = True
        return self.get_current_question_text()

    def get_current_question_text(self):
        if self.current_index >= len(self.questions): return None
        q = self.questions[self.current_index]
        options = [q['option_1'], q['option_2'], q['option_3'], q['option_4']]
        random.shuffle(options)
        labels = ['A', 'B', 'C', 'D']
        self.answers_map = {l: opt for l, opt in zip(labels, options)}

        txt = f"📝 **Soal {self.current_index + 1}/{len(self.questions)}**\n*{q['question_desc']}*\n\n"
        for l in labels: txt += f"{l}. {self.answers_map[l]}\n"
        txt += "\n*(Jawab dengan ketik A, B, C, atau D)*"
        return txt

    def submit_answer(self, user_answer):
        user_answer = user_answer.upper().strip()
        if user_answer not in ['A', 'B', 'C', 'D']: return False, "Jawab pake huruf A, B, C, atau D aja ya!"

        correct = self.questions[self.current_index]['correct_answer']
        selected = self.answers_map[user_answer]
        is_correct = (selected == correct)
        if is_correct: self.score += 1; feedback = "🎉 **Mantap! Benar!**"
        else: feedback = f"❌ **Kurang tepat.** Jawabannya: {correct}"

        self.current_index += 1
        return is_correct, feedback

    def get_result(self):
        pct = (self.score / len(self.questions)) * 100
        lvl = "Beginner" if pct < 40 else "Intermediate" if pct < 80 else "Advanced"
        return {'score': self.score, 'total': len(self.questions), 'level': lvl, 'message': "Semangat!"}

# ==========================================
# 2. HELPER ROADMAP (JANGAN DIUBAH)
# ==========================================
# ==========================================
# 2. HELPER ROADMAP (FIXED LOGIC & VISUAL)
# ==========================================
def generate_visual_roadmap(level_str, topic="General", completed_courses=[], current_course_name=None):
    """
    Visual Roadmap v3: Numbered List + Bold Current Course
    """
    path_row = df_learning_paths[df_learning_paths['learning_path_name'].str.contains(topic, case=False, na=False)]
    if path_row.empty: path_id = 1; path_name = "General Tech"
    else: path_id = path_row.iloc[0]['learning_path_id']; path_name = path_row.iloc[0]['learning_path_name']

    courses = df_courses[df_courses['learning_path_id'] == path_id].sort_values('course_level_str')

    lvl_map = {"Dasar":1, "Pemula":2, "Menengah":3, "Mahir":4, "Profesional":5, "Beginner":1, "Intermediate":3, "Advanced":5}
    user_lvl_num = lvl_map.get(level_str, 1)
    if "Level" in level_str:
        try: user_lvl_num = int(level_str.replace("Level ", ""))
        except: pass

    # PEMISAH PENDEK (20 Karakter)
    sep = "-" * 20
    roadmap_str = f"\n🗺️ **ROADMAP: {path_name.upper()}**\nStatus: {level_str}\n{sep}\n"
    found_next = False

    for idx, row in enumerate(courses.itertuples(), 1):
        c_name = row.course_name
        c_lvl_num = int(row.course_level_str) if str(row.course_level_str).isdigit() else 0

        prefix = f"{idx}. "; suffix = ""; is_bold = False

        if c_name == current_course_name:
            icon = "🚀"; suffix = " (Sedang Belajar)"; is_bold = True; found_next = True
        elif c_name in completed_courses: icon = "✅"
        elif c_lvl_num < user_lvl_num: icon = "✅"
        elif c_lvl_num == user_lvl_num and not found_next: icon = "👉"; suffix = " (Ambil Ini)"; found_next = True
        elif c_lvl_num > user_lvl_num and not found_next: icon = "👉"; suffix = " (Ambil Ini)"; found_next = True
        else: icon = "🔒"

        course_display = f"**{c_name}**" if is_bold else c_name
        roadmap_str += f"{prefix}{icon} {course_display}{suffix}\n"

    return roadmap_str + sep + "\n"

def get_next_course_recommendation(email):
    try:
        user_prog = df_student_progress[df_student_progress['email'] == email]
        if user_prog.empty: return "Yuk mulai belajar course dasar!"
        last_course = user_prog.iloc[0]['course_name']
        c_info = df_courses[df_courses['course_name'] == last_course]
        if c_info.empty: return f"Lanjutkan *{last_course}*!"
        lp_id = c_info.iloc[0]['learning_path_id']
        path_courses = df_courses[df_courses['learning_path_id'] == lp_id].sort_values('course_level_str')['course_name'].tolist()

        if last_course in path_courses:
            idx = path_courses.index(last_course)
            if idx + 1 < len(path_courses):
                next_c = path_courses[idx+1]
                info = df_courses[df_courses['course_name'] == next_c].iloc[0]
                return f"\n🚀 **MISI SELANJUTNYA:**\n👉 **{next_c}**\n📊 Level: {info.get('course_level_str','-')} | ⏳ Estimasi: {info.get('hours_to_study','-')} Jam\n\nYuk, upgrade skill kamu di sini!"
            else: return "🎓 **LUAR BIASA!** Kamu sudah menamatkan path ini."
        return f"Teruskan *{last_course}*!"
    except: return "Semangat belajarnya!"

In [ ]:
def learning_buddy_chat(user_input, user_email=None):
    print(f"\n{'='*60}\nUSER: {user_input} | EMAIL: {user_email}\n{'='*60}\n")

    # 1. Init Memory
    mem = init_user_memory(user_email)
    quiz_sess = mem.get('quiz_session') if mem else None

    if quiz_sess and quiz_sess.active:
        try:
            is_correct, feedback = quiz_sess.submit_answer(user_input)
            feedback = feedback.replace("bestie", "kak").replace("Mantap!", "Mantap kak!")

            next_q = quiz_sess.get_current_question_text()
            if next_q:
                resp_text = f"{feedback}\n{next_q}"
            else:
                # Kuis Selesai
                res = quiz_sess.get_result()
                quiz_sess.active = False

                # Generate Roadmap Visual kalau kuis selesai
                roadmap_viz = ""
                if user_email:
                    try:
                        topic = quiz_sess.topic if quiz_sess.topic else "General Tech"
                        u_prog = df_student_progress[df_student_progress['email'] == user_email]
                        if not u_prog.empty:
                            # Logic visual roadmap singkat
                            curr_c = u_prog.iloc[0]['course_name']
                            is_grad = u_prog.iloc[0]['is_graduated']
                            c_info = df_courses[df_courses['course_name'] == curr_c]
                            if not c_info.empty:
                                lp_id = c_info.iloc[0]['learning_path_id']
                                lp_name = df_learning_paths[df_learning_paths['learning_path_id'] == lp_id].iloc[0]['learning_path_name']
                                user_lvl = c_info.iloc[0]['course_level_str']
                                roadmap_viz = generate_visual_roadmap(f"Level {user_lvl}", lp_name, [curr_c] if is_grad else [], curr_c if not is_grad else None)
                    except: pass

                resp_text = f"🏁 **QUIZ SELESAI!**\nNilai: {res['score']}/{res['total']}\nLevel: **{res['level']}**\n{res['message']}\n\n{roadmap_viz}\n\nAku sudah buatkan roadmap di atas sesuai level kamu. Mau rekomendasi detail?"

            if user_email: add_to_memory(user_email, user_input, resp_text)
            return {'response': resp_text, 'assessment': {}, 'tutorials': [], 'roadmap': []}
        except Exception as e:
            print(f"Quiz Error: {e}")
            if quiz_sess: quiz_sess.active = False
            return {'response': "Maaf, kuis error. Kita reset ya.", 'assessment': {}, 'tutorials': [], 'roadmap': []}

    if any(k in user_input.lower() for k in ['quiz', 'kuis', 'tes skill', 'uji kemampuan']):
        text = user_input.lower()
        topic = None
        # Simple topic detection
        if "android" in text: topic = "Android"
        elif "cloud" in text or "gcp" in text: topic = "Cloud Computing"
        elif "data" in text: topic = "Data"
        elif "ios" in text: topic = "iOS"
        elif "machine" in text or "ai" in text: topic = "Machine Learning"
        elif "multi" in text or "flutter" in text: topic = "Multi Platform"
        elif "web" in text: topic = "Web"

        new_quiz = QuizSession(topic)
        q1_text = new_quiz.start(df_quiz_questions, specific_topic=topic)

        # Save to memory
        if mem: mem['quiz_session'] = new_quiz

        topic_disp = topic if topic else "Campuran"
        resp_text = f"Gas! Quiz topik **{topic_disp}**. 10 soal, jawab A/B/C/D.\n{q1_text}"

        if user_email: add_to_memory(user_email, user_input, resp_text)
        return {'response': resp_text, 'assessment': {}, 'tutorials': [], 'roadmap': []}

    # 1. Assessment & Context Building
    assessment = assess_user(user_input)
    tutorials = search_tutorials(user_input, top_k=5)

    roadmap_data = []
    if assessment.get('role'):
        roadmap_data = generate_roadmap(assessment['role'], assessment['level'], 4)
    elif assessment.get('skills'):
        roadmap_data = generate_skill_based_roadmap(assessment['skills'], 4)

    # 2. Build Prompt (Pake fungsi Context Pro Max yang tadi)
    prompt_context = build_conversational_context(
        user_input, assessment, tutorials, roadmap_data, user_email
    )

    # 3. Generate LLM Response
    try:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)

        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": prompt_context}],
            temperature=0.7,
            max_tokens=850
        )

        ai_response = chat_completion.choices[0].message.content.strip()

        if user_email:
            add_to_memory(user_email, user_input, ai_response)

        return {
            'response': ai_response,
            'assessment': assessment,
            'tutorials': tutorials,
            'roadmap': roadmap_data
        }

    except Exception as e:
        print(f"Error Groq: {e}")
        return {
            'response': "Waduh, otak AI ku lagi maintenance sebentar. Coba tanya lagi ya! 🤖",
            'assessment': {}, 'tutorials': [], 'roadmap': []
        }

In [ ]:
import os
if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("Key set ✓")
else:
    print("❌ ERROR: GROQ_API_KEY tidak ditemukan!")

In [ ]:
from groq import Groq
import time

def get_groq_client():
    """Initialize Groq client untuk Llama 3"""
    return Groq(api_key=GROQ_API_KEY)

print("✅ Groq client ready!")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

def generate_fallback_response(assessment, tutorials, roadmap):
    """
    Fallback response - tetap pakai Dicoding + external jika perlu
    """

    role = assessment.get('role', '')
    level = assessment.get('level', 'Beginner')
    skills = assessment.get('skills', [])

    response = "📚 **Rekomendasi Belajar untuk Kamu**\n\n"

    # Intro
    if role:
        response += f"🎯 Target: **{role}** (Level {level})\n\n"
    elif skills:
        response += f"💡 Skills: **{', '.join(skills[:3])}**\n\n"

    # Dicoding resources
    if tutorials:
        response += "**Dari Dicoding:**\n"
        for i, tut in enumerate(tutorials[:3], 1):
            response += f"{i}. {tut['tutorial']}\n   📖 {tut['course']} • {tut['level']}\n\n"

    if roadmap:
        response += "**Roadmap Belajar:**\n"
        for item in roadmap[:4]:
            response += f"📅 Week {item['week']}: {item['course']} ({item['level']})\n"
        response += "\n"

    # External resources jika data kurang
    if should_use_external_resources(tutorials, roadmap):
        ext_res = get_external_resources(role or (skills[0] if skills else ''))
        if ext_res:
            response += "**Resource Pelengkap:**\n"
            response += "_Karena data Dicoding terbatas, berikut resource tambahan:_\n\n"
            for i, res in enumerate(ext_res, 1):
                response += f"{i}. {res['title']} ({res['type']})\n"
            response += "\n"

    # Tips
    response += "💪 **Strategi Belajar:**\n"
    response += "1. Mulai dari level yang sesuai\n"
    response += "2. Praktik langsung setiap materi\n"
    response += "3. Konsisten belajar rutin\n"
    response += "4. Join komunitas untuk diskusi\n\n"

    response += "Aku di sini kalau butuh teman diskusi! 😊"

    return response

print("✅ Fallback ready (with external resources)!")

In [ ]:
LEVEL_RANK = {
    'Dasar': 1,
    'Pemula': 2,
    'Menengah': 3,
    'Mahir': 4,
    'Profesional': 5
}

def generate_roadmap(role, level, max_courses=6):
    """
    Generate learning roadmap berdasarkan role & level
    STRICTLY from Dicoding LP + Course data
    """

    if not role:
        return []

    try:
        role_clean = clean_text(role)

        # Search in learning path mapping
        df_lp_copy = df_lp_mapping.copy()
        df_lp_copy['lp_clean'] = df_lp_copy['learning_path_name'].apply(clean_text)

        # Find relevant learning paths
        relevant = df_lp_copy[df_lp_copy['lp_clean'].str.contains(role_clean, na=False)].copy()

        # If no exact match, use broader search
        if relevant.empty:
            # Search by keywords
            role_keywords = {
                'backend': ['backend', 'server', 'api', 'node', 'python'],
                'frontend': ['frontend', 'react', 'web', 'javascript'],
                'android': ['android', 'mobile', 'kotlin'],
                'data': ['data', 'machine learning', 'ai'],
                'devops': ['devops', 'cloud', 'docker']
            }

            for key, keywords in role_keywords.items():
                if any(k in role_clean for k in keywords):
                    relevant = df_lp_copy[df_lp_copy['lp_clean'].str.contains('|'.join(keywords), na=False)].copy()
                    break

            # Still empty? Use all courses
            if relevant.empty:
                relevant = df_lp_copy.copy()

        # Add level ranking
        relevant['level_rank'] = relevant['course_level_str'].map(LEVEL_RANK).fillna(1)

        # Filter by user level
        user_rank = LEVEL_RANK.get(level, 1)
        relevant = relevant[relevant['level_rank'] >= user_rank]

        # Sort and deduplicate
        relevant = relevant.sort_values('level_rank').drop_duplicates('course_name')

        # Generate roadmap
        roadmap = []
        for i, (_, row) in enumerate(relevant.head(max_courses).iterrows(), 1):
            roadmap.append({
                'week': i,
                'course': row['course_name'],
                'level': row.get('course_level_str', 'Dasar'),
                'learning_path': row.get('learning_path_name', '')
            })

        return roadmap

    except Exception as e:
        print(f"❌ Error in generate_roadmap: {str(e)}")
        return []

def generate_skill_based_roadmap(skills, weeks=6):
    """
    Generate roadmap berdasarkan skills list
    STRICTLY from Dicoding courses
    """

    if not skills or len(skills) == 0:
        return []

    try:
        # Use first skill as primary
        skill_query = skills[0].lower()

        # Search in courses
        matches = df_courses[
            df_courses["course_name"].str.lower().str.contains(skill_query, na=False)
        ]

        # If no match, try broader search
        if matches.empty:
            for skill in skills[:3]:
                matches = df_courses[
                    df_courses["course_name"].str.lower().str.contains(skill.lower(), na=False)
                ]
                if not matches.empty:
                    break

        # Generate roadmap
        roadmap = []
        for idx, row in matches.head(weeks).iterrows():
            roadmap.append({
                "week": len(roadmap) + 1,
                "course": row["course_name"],
                "level": row.get("course_level_str", "Pemula")
            })

        return roadmap

    except Exception as e:
        print(f"❌ Error in generate_skill_based_roadmap: {str(e)}")
        return []

print("✅ Enhanced roadmap generation ready!")

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta

class SimpleRateLimiter:
    def __init__(self, max_requests=10, window_seconds=60):
        self.requests = defaultdict(list)
        self.max_requests = max_requests
        self.window_seconds = window_seconds

    def is_allowed(self, user_id="default"):
        now = datetime.now()
        cutoff = now - timedelta(seconds=self.window_seconds)

        # Hapus request lama
        self.requests[user_id] = [
            req for req in self.requests[user_id]
            if req > cutoff
        ]

        # Cek limit
        if len(self.requests[user_id]) >= self.max_requests:
            return False

        self.requests[user_id].append(now)
        return True

    def get_wait_time(self, user_id="default"):
        """Berapa lama user harus tunggu (dalam detik)"""
        if not self.requests[user_id]:
            return 0

        oldest = min(self.requests[user_id])
        wait = (oldest + timedelta(seconds=self.window_seconds) - datetime.now()).total_seconds()
        return max(0, int(wait))

# Inisialisasi rate limiter
rate_limiter = SimpleRateLimiter(max_requests=15, window_seconds=60)

In [ ]:
from datetime import datetime

# ==========================================
# MEMORY MANAGEMENT (FIXED & COMPLETE)
# ==========================================

# Global variable check
if 'conversation_memory' not in globals():
    conversation_memory = {}

def init_user_memory(email):
    """
    Initialize memory user dengan struktur LENGKAP.
    Support: Quiz, Greeting Logic, Chat History, & Stats
    """
    if not email:
        return None

    # 1. Buat Memory Baru (Fresh User)
    if email not in conversation_memory:
        conversation_memory[email] = {
            'history': [],            # Untuk Context RAG
            'messages': [],           # Untuk Log detail (User & Bot)
            'context': None,
            'last_interaction': datetime.now(),
            'total_interactions': 0,  # Counter interaksi
            'greeted': False,         # Flag greeting
            'quiz_session': None      # State buat Quiz
        }

    # 2. SAFETY CHECK (Backward Compatibility)
    # Ini PENTING biar gak crash kalau memory user udah ada tapi format lama (kurang key)
    mem = conversation_memory[email]

    # Cek satu-satu key wajib, kalau gak ada kita suntik default value
    if 'messages' not in mem: mem['messages'] = []
    if 'greeted' not in mem: mem['greeted'] = False
    if 'quiz_session' not in mem: mem['quiz_session'] = None
    if 'total_interactions' not in mem: mem['total_interactions'] = 0
    if 'history' not in mem: mem['history'] = []

    return mem

def add_to_memory(email, user_msg, bot_response):
    """Save conversation ke memory dengan aman"""
    if not email:
        return

    # Panggil init dulu biar yakin struktur lengkap
    memory = init_user_memory(email)

    # Append ke messages
    memory['messages'].append({
        'user': user_msg,
        'bot': bot_response,
        'timestamp': datetime.now().isoformat()
    })

    # Update stats
    memory['last_interaction'] = datetime.now()
    memory['total_interactions'] += 1

    # Append ke history (Format text buat RAG context)
    memory['history'].append(f"User: {user_msg}\nBot: {bot_response}")

def should_greet(text, email):
    """
    Cek apakah perlu greeting (Hanya sekali di awal)
    """
    if not email:
        return False

    memory = init_user_memory(email)

    # ✅ Kalau sudah pernah greet, JANGAN greet lagi
    if memory['greeted']:
        return False

    # Check logic greeting sederhana
    greetings = ['halo', 'hai', 'hi', 'hello', 'hey', 'hallo', 'pagi', 'siang', 'sore', 'malam']
    text_clean = text.lower().strip().replace('!', '').replace('?', '').replace('.', '')
    words = text_clean.split()

    # Dianggap greeting kalau kalimat pendek (<=3 kata) dan mengandung kata sapaan
    is_greeting = (len(words) <= 3 and any(g in words for g in greetings))

    if is_greeting:
        memory['greeted'] = True  # Tandai sudah disapa
        return True

    return False

def is_small_talk(text):
    """Detect small talk tanpa greeting"""
    patterns = [
        'apa kabar', 'bagaimana kabar', 'gimana kabar',
        'lagi ngapain', 'lagi apa', 'gimana',
        'terima kasih', 'makasih', 'thanks', 'thank you',
        'baik baik', 'bagus', 'oke', 'sip', 'mantap'
    ]
    text_lower = text.lower()
    return any(p in text_lower for p in patterns)

def reset_greeting(email):
    """Reset greeting status - untuk testing"""
    if email in conversation_memory:
        conversation_memory[email]['greeted'] = False

print("✅ FIXED: Memory Logic Loaded Successfully!")
print("   - Quiz Session Ready")
print("   - Messages Log Ready")
print("   - Greeted Flag Ready")

In [ ]:
def build_conversational_context(user_input, assessment, tutorials, roadmap, user_email=None):
    """
    ✅ FINAL PRO MAX (FIXED KEYS): Support History, Jadwal, & Visual Roadmap
    """

    # 1. Ambil History Chat
    history_text = ""
    if user_email:
        mem = init_user_memory(user_email)
        last_chats = mem['history'][-3:] if mem['history'] else []
        if last_chats:
            history_text = "\nRIWAYAT CHAT SEBELUMNYA:\n" + "\n".join(last_chats) + "\n"

    role = assessment.get('role', '')
    level = assessment.get('level', 'Beginner')
    skills = assessment.get('skills', [])
    skills_text = ', '.join(skills[:5]) if skills else 'Belum terdeteksi'

    # 2. Progress & Roadmap Data
    detailed_history = None
    visual_roadmap_str = ""
    is_asking_progress = any(k in user_input.lower() for k in ['progress', 'kemajuan', 'history', 'pencapaian', 'roadmap', 'jalur'])

    if user_email and is_asking_progress:
        detailed_history = get_detailed_history(user_email)

        # --- LOGIC VISUAL ROADMAP ---
        try:
            u_prog = df_student_progress[df_student_progress['email'] == user_email]
            if not u_prog.empty:
                curr_c = u_prog.iloc[0]['course_name']
                is_grad = u_prog.iloc[0]['is_graduated']

                c_info = df_courses[df_courses['course_name'] == curr_c]
                if not c_info.empty:
                    lp_id = c_info.iloc[0]['learning_path_id']
                    lp_name = df_learning_paths[df_learning_paths['learning_path_id'] == lp_id].iloc[0]['learning_path_name']

                    completed_list = [curr_c] if is_grad == 1 else []
                    viz_curr = curr_c if is_grad == 0 else None
                    user_lvl_str = c_info.iloc[0]['course_level_str']

                    visual_roadmap_str = generate_visual_roadmap(f"Level {user_lvl_str}", lp_name, completed_list, viz_curr)
        except Exception as e:
            print(f"⚠️ Error generating visual roadmap: {e}")

    # ============================================================
    # START SYSTEM PROMPT
    # ============================================================
    context = f"""Kamu adalah Learning Buddy, asisten belajar AI dari Dicoding.

IDENTITY & BEHAVIOR:
- Teman diskusi yang asik, suportif, dan pinter.
- GAYA BICARA: Casual, pakai "Aku/Kamu", boleh pakai emoji.
- PENTING: Jangan menyapa "Halo/Hai" lagi jika di RIWAYAT CHAT sudah ada sapaan.

🚨 GUARDRAILS (ATURAN KERAS):
1. Jika input user hanya kata tunggal tidak bermakna seperti "test", "tes", "cek", "p", "halo": JANGAN BUATKAN ROADMAP. Cukup jawab: "Sip, Learning Buddy aman dan siap! Mau belajar topik apa hari ini? 😊"
2. Jika input user tidak jelas topiknya, tanya balik.

CAPABILITIES:
1. Menjawab pertanyaan teknis & progress belajar.
2. Memberikan rekomendasi course.
3. GENERAL KNOWLEDGE: Boleh jawab tren pasar/gaji di luar data Dicoding.
4. SCHEDULING: Bisa buatkan jadwal belajar mingguan.

USER CONTEXT:
- Role Target: {role if role else 'Belum terdeteksi'}
- Level Terdeteksi: {level}
- Skill Interest: {skills_text}
{history_text}
USER INPUT SAAT INI: "{user_input}"
"""

    # ============================================================
    # DATA INJECTION (FIXED KEYS HERE)
    # ============================================================
    if detailed_history:
        # Perbaikan kunci dictionary di sini:
        # 'course' bukan 'current_course'
        # 'completed' bukan 'completed_tutorials'
        # 'total' bukan 'active_tutorials'
        # 'pct' bukan 'progress_percentage'
        # 'milestone' bukan 'next_milestone'

        context += f"""
DATA PROGRESS USER (REAL-TIME):
- Course Aktif: {detailed_history['course']}
- Selesai: {detailed_history['completed']}/{detailed_history['total']} Modul ({detailed_history['pct']}%)
- Status: {'LULUS' if detailed_history['is_graduated'] else 'ON GOING'}
- Next Step: {detailed_history['milestone']}
"""
        if visual_roadmap_str:
            context += f"""
VISUAL ROADMAP USER (TAMPILKAN INI JIKA DIMINTA ROADMAP/PROGRESS):
{visual_roadmap_str}
"""
        context += """
INSTRUKSI PROGRESS:
- Bahas progress dia secara spesifik berdasarkan data di atas.
- Jika ada 'VISUAL ROADMAP USER', WAJIB sertakan dalam jawabanmu.
"""

    if tutorials:
        context += "\nREKOMENDASI COURSE DICODING:\n"
        for i, tut in enumerate(tutorials[:5], 1):
            context += f"{i}. {tut['tutorial']} ({tut['course']})\n"

    if roadmap:
        context += "\nROADMAP KARIER:\n"
        for item in roadmap[:6]:
            context += f"- {item.get('week', 'Step')}: {item.get('course', 'Materi')}\n"

    context += """
INSTRUKSI JAWABAN AKHIR:
1. Prioritaskan data Dicoding di atas.
2. Jika data Dicoding tidak ada/kurang relevan (misal trend/gaji/teknologi baru), GUNAKAN PENGETAHUAN UMUM.
3. Jika diminta jadwal, buatkan tabel/poin jadwal mingguan.
"""
    return context

In [ ]:
SUB_SKILLS_DATABASE = {
    'Python': {
        'Basic Syntax': ['variables', 'data types', 'operators', 'conditionals', 'loops'],
        'Data Structures': ['list', 'tuple', 'dictionary', 'set'],
        'OOP': ['class', 'inheritance', 'polymorphism', 'encapsulation'],
        'Advanced': ['decorators', 'generators', 'context managers', 'async']
    },
    'Machine Learning': {
        'Fundamentals': ['supervised learning', 'unsupervised learning', 'regression', 'classification'],
        'Algorithms': ['decision tree', 'random forest', 'svm', 'neural networks'],
        'Tools': ['scikit-learn', 'tensorflow', 'pytorch', 'keras'],
        'Deployment': ['model serving', 'mlops', 'monitoring']
    },
    'React': {
        'Basics': ['jsx', 'components', 'props', 'state'],
        'Hooks': ['useState', 'useEffect', 'useContext', 'custom hooks'],
        'Advanced': ['redux', 'performance', 'testing', 'ssr']
    },
    'JavaScript': {
        'Fundamentals': ['variables', 'functions', 'arrays', 'objects'],
        'ES6+': ['arrow functions', 'destructuring', 'spread operator', 'promises'],
        'Async': ['callbacks', 'promises', 'async/await', 'fetch api'],
        'Advanced': ['closures', 'prototypes', 'event loop', 'modules']
    },
    'Backend': {
        'Fundamentals': ['http', 'rest api', 'database', 'authentication'],
        'Node.js': ['express', 'middleware', 'routing', 'error handling'],
        'Database': ['sql', 'nosql', 'orm', 'migrations'],
        'Deployment': ['docker', 'ci/cd', 'monitoring', 'scaling']
    }
}

def get_sub_skills(skill_name):
    """
    LAYER 3: Get sub-skills untuk detail assessment
    Sesuai soal: Uji level detail per sub-skill
    """
    for skill, subs in SUB_SKILLS_DATABASE.items():
        if skill.lower() in skill_name.lower():
            return subs
    return None

def generate_sub_skill_questions(skill_name):
    """Generate pertanyaan untuk assess sub-skill (minimal 6 sub-skill)"""
    sub_skills = get_sub_skills(skill_name)

    if not sub_skills:
        return None

    questions = {
        'skill': skill_name,
        'categories': {},
        'total_sub_skills': 0
    }

    for category, items in sub_skills.items():
        questions['categories'][category] = {
            'sub_skills': items,
            'questions': [f"Seberapa familiar kamu dengan {item}?" for item in items],
            'level': 'Not Tested'
        }
        questions['total_sub_skills'] += len(items)

    return questions

def calculate_skill_level(responses):
    """
    Calculate level berdasarkan sub-skill assessment
    responses: dict {sub_skill: score} where score is 0-100
    """
    if not responses:
        return 'Beginner'

    avg_score = sum(responses.values()) / len(responses)

    if avg_score >= 70:
        return 'Advanced'
    elif avg_score >= 40:
        return 'Intermediate'
    else:
        return 'Beginner'

print("✅ Sub-skill assessment (Layer 3) ready!")

In [ ]:
def update_roadmap_based_on_progress(user_email, current_roadmap):
    """
    Adaptive roadmap: update based on user progress
    """
    progress = get_detailed_history(user_email)

    if not progress:
        return current_roadmap

    # Calculate completion rate
    completion_rate = progress.get('progress_percentage', 0)

    updated_roadmap = []

    for item in current_roadmap:
        # Skip courses that might be too easy if user is progressing well
        if completion_rate > 70 and item['level'] == 'Dasar':
            continue

        # Add accelerated flag for fast learners
        if completion_rate > 50:
            item['suggestion'] = '⚡ Fast track available - bisa skip ke advance'

        updated_roadmap.append(item)

    # Add next level courses if progressing well
    if completion_rate > 80 and len(updated_roadmap) < 6:
        # Get more advanced courses
        advanced_courses = df_courses[
            df_courses['course_level_str'].isin(['Mahir', 'Profesional'])
        ].head(2)

        for idx, row in advanced_courses.iterrows():
            updated_roadmap.append({
                'week': len(updated_roadmap) + 1,
                'course': row['course_name'],
                'level': row.get('course_level_str', 'Mahir'),
                'suggestion': '🎯 Next level challenge'
            })

    return updated_roadmap

In [ ]:
EXTERNAL_RESOURCES = {
    'Python': [
        {'title': 'Python.org Official Docs', 'url': 'https://docs.python.org/', 'type': 'documentation'},
        {'title': 'Real Python Tutorials', 'url': 'https://realpython.com/', 'type': 'tutorial'},
        {'title': 'Python Discord Community', 'url': 'https://pythondiscord.com/', 'type': 'community'}
    ],
    'Machine Learning': [
        {'title': 'Kaggle Learn', 'url': 'https://www.kaggle.com/learn', 'type': 'practice'},
        {'title': 'Papers with Code', 'url': 'https://paperswithcode.com/', 'type': 'research'},
    ],
    'React': [
        {'title': 'React Official Docs', 'url': 'https://react.dev/', 'type': 'documentation'},
        {'title': 'React Dev Community', 'url': 'https://dev.to/t/react', 'type': 'community'}
    ],
    'Backend': [
        {'title': 'MDN Web Docs', 'url': 'https://developer.mozilla.org/', 'type': 'documentation'},
        {'title': 'Backend Dev Roadmap', 'url': 'https://roadmap.sh/backend', 'type': 'roadmap'}
    ],
    'JavaScript': [
        {'title': 'MDN JavaScript Guide', 'url': 'https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide', 'type': 'documentation'},
        {'title': 'JavaScript.info', 'url': 'https://javascript.info/', 'type': 'tutorial'}
    ],
    'Android': [
        {'title': 'Android Developers', 'url': 'https://developer.android.com/', 'type': 'documentation'},
        {'title': 'Kotlin Docs', 'url': 'https://kotlinlang.org/docs/', 'type': 'documentation'}
    ]
}

def should_use_external_resources(tutorials, roadmap):
    """
    Decide when to use external resources
    """
    if len(tutorials) < 3 or len(roadmap) < 3:
        return True
    return False


def get_external_resources(skill_or_role):
    """Get external resources based on skill/role"""
    resources = []
    skill_lower = skill_or_role.lower()

    if any(k in skill_lower for k in ['python', 'py']):
        resources.extend(EXTERNAL_RESOURCES.get('Python', []))

    if any(k in skill_lower for k in ['machine learning', 'ml', 'ai', 'data science']):
        resources.extend(EXTERNAL_RESOURCES.get('Machine Learning', []))

    if any(k in skill_lower for k in ['react', 'vue', 'angular', 'frontend', 'front-end']):
        resources.extend(EXTERNAL_RESOURCES.get('React', []))

    if any(k in skill_lower for k in ['backend', 'back-end', 'api', 'server']):
        resources.extend(EXTERNAL_RESOURCES.get('Backend', []))

    if any(k in skill_lower for k in ['javascript', 'js', 'typescript']):
        resources.extend(EXTERNAL_RESOURCES.get('JavaScript', []))

    if any(k in skill_lower for k in ['android', 'kotlin', 'mobile']):
        resources.extend(EXTERNAL_RESOURCES.get('Android', []))

    # Remove duplicates
    seen = set()
    unique_resources = []
    for r in resources:
        if r['url'] not in seen:
            seen.add(r['url'])
            unique_resources.append(r)

    return unique_resources[:3]

def format_external_resources(resources):
    """Format external resources untuk display"""
    if not resources:
        return ""

    msg = "\n\n📚 **Resource Tambahan (di luar Dicoding):**\n"
    msg += "_Karena data Dicoding terbatas untuk topik ini, berikut resource pelengkap:_\n\n"
    for i, res in enumerate(resources, 1):
        msg += f"{i}. [{res['title']}]({res['url']}) - {res['type']}\n"

    return msg

print("✅ External resources ready (digunakan saat data Dicoding kurang)!")

In [ ]:
def handle_api_chat(message, user_email=None):
    """
    Updated API handler dengan improved conversation
    """
    try:
        result = learning_buddy_chat(
            user_input=message,
            user_email=user_email
        )

        return {
            'status': 'success',
            'data': {
                'response': result['response'],
                'assessment': result.get('assessment', {}),
                'tutorials': result.get('tutorials', []),
                'roadmap': result.get('roadmap', []),
                'history': result.get('history', None)  # NEW: Include detailed history
            },
            'timestamp': datetime.now().isoformat()
        }

    except Exception as e:
        return {
            'status': 'error',
            'data': {
                'response': "Maaf, ada error. Coba lagi ya! 🙏",
                'assessment': {},
                'tutorials': [],
                'roadmap': []
            },
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

print("✅ Improved Learning Buddy loaded!")
print("\nKey improvements:")
print("1. ✅ No repetitive greetings")
print("2. ✅ Detailed learning history with timeline")
print("3. ✅ Flexible small talk like GPT/Gemini")
print("4. ✅ Conversational memory per user")
print("\nRun test_improved_conversations() to test!")

In [ ]:
from pyngrok import ngrok
import os

# 1. Matikan proses ngrok yang nyangkut
print("🔪 Killing zombie ngrok processes...")
ngrok.kill()

# 2. Opsional: Kill proses system level biar bersih total (Linux/Colab)
os.system("pkill ngrok")

# 3. Baru deh connect ulang (lanjut ke kode lu yang biasa)
print("✅ Ngrok bersih! Siap menyalakan server baru...")

In [ ]:
# ====================================================================
# 🚨 PENTING: PASTIKAN CELL MUAT KEY (Cell 2) SUDAH DIJALANKAN DULU!
# ====================================================================

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from threading import Thread
import time
import os
import traceback

os.system("fuser -k 8000/tcp")

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    user_input: str
    user_email: Optional[str] = None

@app.post("/api/chat")
async def chat_endpoint(data: ChatRequest):
    print(f"\n📨 Request Masuk:")
    print(f"   Input: {data.user_input}")
    print(f"   Email: {data.user_email}")

    try:
        # Panggil fungsi utama learning_buddy_chat
        response_data = learning_buddy_chat(
            user_input=data.user_input,
            user_email=data.user_email
        )

        print(f"✅ Response sukses generated")

        return {
            "status": "success",
            "data": {
                "response": response_data
            }
        }

    except Exception as e:
        print("❌ ERROR DI SERVER:")
        traceback.print_exc()
        return {
            "status": "error",
            "data": {
                "response": {
                    "response": "Maaf, bot sedang gangguan sistem. Coba lagi ya! 🙏",
                    "assessment": {}, "tutorials": [], "roadmap": []
                }
            }
        }

@app.get("/")
def root():
    return {"msg": "✅ Learning Buddy Server Online!"}

print("🚀 MENYALAKAN SERVER...")
nest_asyncio.apply()

ngrok.kill()
time.sleep(1)

if 'NGROK_AUTH_TOKEN' in globals() and NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ Ngrok Auth Token berhasil di-set dari environment.")
else:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN tidak ditemukan di variabel global!")
    print("Pastikan Anda sudah upload file .env di Cell awal.")

def start_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = Thread(target=start_uvicorn, daemon=True)
thread.start()

time.sleep(3)

try:
    url = ngrok.connect(8000).public_url
    print(f"\n✅ URL NGROK BARU: {url}")
    print("👉 Copy URL ini ke frontend/script.js sekarang!")
except Exception as e:
    print(f"Ngrok Error: {e}")

In [ ]:
def test_improved_conversations():
    """Test various conversation scenarios"""

    test_cases = [
        {
            "name": "First Greeting",
            "input": "Halo!",
            "email": "irfan.santoso53@example.com"
        },
        {
            "name": "Second Greeting (should not repeat)",
            "input": "Hi lagi nih",
            "email": "irfan.santoso53@example.com"
        },
        {
            "name": "Small Talk",
            "input": "Apa kabar?",
            "email": "irfan.santoso53@example.com"
        },
        {
            "name": "Progress Question with History",
            "input": "Progress belajarku gimana? Ada tips gak biar lebih cepet?",
            "email": "irfan.santoso53@example.com"
        },
        {
            "name": "Learning Request",
            "input": "Aku mau belajar Python untuk data science",
            "email": "irfan.santoso53@example.com"
        },
        {
            "name": "Thank You",
            "input": "Makasih ya!",
            "email": "irfan.santoso53@example.com"
        }
    ]

    print("\n" + "="*70)
    print("🧪 TESTING IMPROVED CONVERSATIONS")
    print("="*70 + "\n")

    for i, test in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"TEST {i}: {test['name']}")
        print('='*70)

        result = learning_buddy_chat(test['input'], test['email'])
        print(f"\n✅ Response preview:")
        print(result['response'][:300] + "..." if len(result['response']) > 300 else result['response'])

        import time
        time.sleep(2)  # Avoid rate limit

In [ ]:
# Hapus memory lama biar gak crash
conversation_memory = {}
print("🧹 Memory berhasil di-wipe! Siap untuk testing ulang.")

In [ ]:
test_improved_conversations()

In [ ]:
def test_learning_buddy_requirements():
    """
    Test sesuai requirements soal:
    1. Multi-layer assessment
    2. Progress tracking
    3. Personal assistant
    4. Motivasi & strategi
    5. External resources jika perlu
    """

    test_cases = [
        {
            "name": "1. First Greeting (one-time only)",
            "input": "Halo!",
            "email": "test@example.com",
            "expect": "Greeting + explain role as personal assistant"
        },
        {
            "name": "2. Multi-Layer Assessment - Job Role Detection",
            "input": "Aku mau jadi Backend Developer, sudah paham JavaScript tapi masih pemula di Node.js",
            "email": "test@example.com",
            "expect": "Role: Backend, Skills: JavaScript/Node.js, Level: Intermediate"
        },
        {
            "name": "3. Progress Tracking & Insight",
            "input": "Progress belajarku gimana? Skill apa yang sudah berkembang minggu ini?",
            "email": "sari.halim271@example.com",
            "expect": "Real progress data + skill insight + motivasi"
        },
        {
            "name": "4. Personalized Roadmap (6 courses)",
            "input": "Bikinin roadmap lengkap untuk jadi AI Engineer dong",
            "email": "test2@example.com",
            "expect": "6-week roadmap from Dicoding"
        },
        {
            "name": "5. Learning Strategy Advice",
            "input": "Aku sering stuck pas belajar Machine Learning. Tips biar lebih cepat paham?",
            "email": "test3@example.com",
            "expect": "Personal advice + strategi belajar + motivasi"
        },
        {
            "name": "6. Teman Diskusi - Casual Question",
            "input": "Menurutmu lebih baik fokus backend atau frontend dulu?",
            "email": "test@example.com",
            "expect": "Diskusi natural + pertimbangan + saran"
        },
        {
            "name": "7. External Resources (data Dicoding terbatas)",
            "input": "Aku mau belajar Rust programming language",
            "email": "test4@example.com",
            "expect": "Dicoding (jika ada) + external resources karena topik spesifik"
        },
        {
            "name": "8. Sub-Skill Assessment Request",
            "input": "Aku sudah bisa Python, tapi bingung level aku sampai mana. Tes aku dong!",
            "email": "test@example.com",
            "expect": "Sub-skill questions untuk Python"
        }
    ]

    print("\n" + "="*70)
    print("🧪 TEST SESUAI REQUIREMENTS SOAL")
    print("="*70 + "\n")

    results = []

    for i, test in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"TEST {i}: {test['name']}")
        print(f"Expected: {test['expect']}")
        print('='*70)

        try:
            result = learning_buddy_chat(test['input'], test['email'])

            print(f"\n✅ Response:")
            print(result['response'][:400] + "..." if len(result['response']) > 400 else result['response'])

            print(f"\n📊 Data:")
            print(f"   Assessment: {result['assessment']}")
            print(f"   Tutorials: {len(result.get('tutorials', []))}")
            print(f"   Roadmap: {len(result.get('roadmap', []))} weeks")

            results.append({"test": test['name'], "status": "PASS"})

        except Exception as e:
            print(f"\n❌ FAILED: {str(e)}")
            results.append({"test": test['name'], "status": "FAIL"})

        import time
        time.sleep(3)  # Avoid rate limit

    # Summary
    print(f"\n{'='*70}")
    print("📊 TEST SUMMARY")
    print('='*70)
    passed = sum(1 for r in results if r['status'] == 'PASS')
    print(f"Passed: {passed}/{len(results)}")
    for r in results:
        status_icon = "✅" if r['status'] == "PASS" else "❌"
        print(f"{status_icon} {r['test']}")
    print('='*70)

test_learning_buddy_requirements()

In [ ]:
# ===== TESTING FUNCTION =====
def test_chat():
    """Test function untuk debugging"""
    test_messages = [
        "Saya ingin belajar python",
        "Aku pemula mau jadi backend developer",
        "Bagaimana cara membuat aplikasi android?"
    ]

    print("\n" + "="*60)
    print("🧪 TESTING CHAT FUNCTION")
    print("="*60 + "\n")

    for msg in test_messages:
        print(f"📨 Test: {msg}")
        result = learning_buddy_chat(msg)
        print(f"✅ Response: {result['response'][:200]}...\n")
        time.sleep(2)  # Delay untuk avoid rate limit

# Uncomment untuk test
# test_chat()


print("\n" + "="*60)
print("✅ ALL FUNCTIONS LOADED!")
print("="*60)
print("\n🔧 Configuration:")
print(f"   - API Key: {GROQ_API_KEY[:20]}...")
print(f"   - Tutorials: {len(df_full)} items")
print(f"   - FAISS Index: {faiss_index.ntotal} vectors")
print(f"   - Rate Limit: 15 requests/minute")
print("\n💡 Ready to start server!")

In [ ]:
def test_comprehensive():
    """Test lengkap untuk semua fitur"""

    print("\n" + "="*70)
    print("🧪 COMPREHENSIVE SYSTEM TEST")
    print("="*70 + "\n")

    test_cases = [
        {
            "name": "Progress Tracking (dengan email)",
            "input": "Saya Rafi, progress saya sampai mana ya?",
            "email": "rafi.santoso5@example.com",
            "check": lambda r: "Sari" in r['response'] or "progress" in r['response'].lower()
        },
        {
            "name": "Role Detection - Data Scientist",
            "input": "Aku mau jadi data scientist dan lagi ambil Microsoft Fabric",
            "email": None,
            "check": lambda r: r['assessment'].get('role') == 'Data Scientist'
        },
        {
            "name": "Roadmap Generation - Python",
            "input": "Bikinin roadmap belajar Python dari dasar dong",
            "email": None,
            "check": lambda r: len(r.get('roadmap', [])) > 0
        },
        {
            "name": "Tutorial Search - React",
            "input": "Aku mau belajar React JS",
            "email": None,
            "check": lambda r: len(r.get('tutorials', [])) > 0
        },
    ]

    passed = 0
    failed = 0

    for i, test in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"TEST {i}: {test['name']}")
        print('='*70)

        try:
            result = learning_buddy_chat(test['input'], test['email'])

            # Check if test passed
            if test['check'](result):
                print(f"✅ TEST {i} PASSED")
                print(f"   Response preview: {result['response'][:120]}...")
                passed += 1
            else:
                print(f"⚠️ TEST {i} PASSED but check failed")
                print(f"   Response: {result['response'][:120]}...")
                passed += 1  # Still count as passed if no crash

        except Exception as e:
            print(f"❌ TEST {i} FAILED")
            print(f"   Error: {str(e)}")
            failed += 1

    print(f"\n{'='*70}")
    print(f"📊 RESULTS: {passed} passed, {failed} failed out of {len(test_cases)} tests")
    print('='*70)

    return passed, failed

In [ ]:
print("\n✅ All functions loaded successfully!")
print("Run test_fixed_system() to test the system")

In [ ]:
# Test lengkap
test_comprehensive()

In [ ]:
# 🚑 CELL DIAGNOSA DARURAT (DEBUGGING MODE)
import traceback

print("🔍 SEDANG MEMERIKSA STATUS SYSTEM...")

# 1. Cek Data Frame
try:
    print(f"✅ Data Tutorials: {len(df_full)} rows")
    print(f"✅ Data Student Progress: {len(df_student_progress)} rows")
except NameError:
    print("❌ ERROR: Data belum ke-load! Run cell 'Loading Excel Data' di atas.")

# 2. Cek API Key
try:
    if not GROQ_API_KEY or GROQ_API_KEY == "gsk_...": # Cek kalo masih default/kosong
        print("❌ ERROR: GROQ_API_KEY belum diisi atau masih default!")
    else:
        print(f"✅ API Key terdeteksi: {GROQ_API_KEY[:5]}...")
except NameError:
    print("❌ ERROR: Variable GROQ_API_KEY gak ketemu!")

# 3. TEST SIMULASI CHAT (PENTING!)
print("\n🧪 MENCOBA SIMULASI CHAT (Bypass Server)...")
try:
    # Kita pake input yang sama persis kayak di web lu tadi
    test_input = "Cek progress saya"
    test_email = "sari.halim271@example.com"

    print(f"📨 Input: {test_input}")
    print(f"📧 Email: {test_email}")

    # Panggil fungsi langsung
    result = learning_buddy_chat(test_input, user_email=test_email)

    print("\n✅ HASIL SUKSES (Gak Crash):")
    print(result['response'][:200] + "...")

except Exception as e:
    print("\n❌ KETEMU MASALAHNYA! (Crash di sini):")
    print(f"👉 Error Type: {type(e).__name__}")
    print(f"👉 Error Message: {str(e)}")
    print("\nTraceback Lengkap:")
    traceback.print_exc()

In [ ]:
import random
import time

# Email tester
test_email = "rafi.santoso5@example.com"

# 1. Reset Memory dulu biar bersih
conversation_memory = {}
print("🧹 Memory Reset. Mulai Test Full Loop...")

# 2. Trigger Quiz
print("\n[USER]: Bro mau quiz Web dong")
resp = learning_buddy_chat("Bro mau quiz Web dong", test_email)
print(f"[BOT]:\n{resp['response']}")

# 3. Loop Jawab Otomatis sampe kelar
# Kita loop max 12 kali (jaga2), tapi harusnya di no 10 dia stop
for i in range(1, 12):
    # Simulasi mikir bentar
    time.sleep(1)

    # Pilih jawaban ngasal (Biar seru, kadang bener kadang salah)
    jawaban_ngasal = random.choice(['A', 'B', 'C', 'D'])

    print(f"\n{'='*20} SOAL KE-{i} {'='*20}")
    print(f"[USER]: {jawaban_ngasal}")

    # Kirim jawaban ke bot
    resp = learning_buddy_chat(jawaban_ngasal, test_email)
    bot_text = resp['response']

    print(f"[BOT]:\n{bot_text}")

    # Cek apakah udah selesai?
    if "QUIZ SELESAI" in bot_text:
        print("\n✅ NAH KAN! MUNCUL ROADMAP & LEVELNYA! 🎉")
        break

In [ ]:
# ==========================================
# 🧪 TES FITUR PROGRESS & ADAPTIVE ROADMAP
# ==========================================

# Kita reset memory biar fresh
conversation_memory = {}
print("🧹 Memory Reset. Siap testing progress...\n")

# --- KASUS 1: USER LAGI BELAJAR (Rafi Santoso) ---
# Di CSV, Rafi lagi belajar 'Belajar Fundamental Aplikasi Android' (Progress 55 tutorial)
email_rafi = "rafi.santoso5@example.com"
chat_rafi = "Progress gue gimana bro? Terus abis ini ambil apa?"

print(f"👤 TES 1: Rafi Santoso (Status: In Progress)")
print(f"   Input: '{chat_rafi}'")
print("-" * 50)

resp_rafi = learning_buddy_chat(chat_rafi, email_rafi)
print(f"[BOT RESPONSE]:\n{resp_rafi['response']}")


print("\n" + "="*60 + "\n")


# --- KASUS 2: USER SUDAH LULUS (Sari Kusuma) ---
# Di CSV, Sari udah lulus 'Belajar Fundamental Aplikasi Android' (Is Graduated = 1)
# Harusnya bot nyaranin course LEVEL LANJUTANNYA (Intermediate/Expert)
email_sari = "sari.kusuma6@example.com"
chat_sari = "Cek progress dong, sekalian minta saran next step."

print(f"👤 TES 2: Sari Kusuma (Status: Graduated)")
print(f"   Input: '{chat_sari}'")
print("-" * 50)

resp_sari = learning_buddy_chat(chat_sari, email_sari)
print(f"[BOT RESPONSE]:\n{resp_sari['response']}")